## Setting up Configuration

In [0]:
%run ./00_setup_config

## Reading data from raw folder

In [0]:
from pyspark.sql.functions import col

try:
    df_raw = spark.read.option("multiline", "true").json(raw_path)
except Exception as e:
    print(f"Error reading JSON: {e}")

df_raw.show(truncate=False)

## Giving Column names + Loading time

In [0]:
from pyspark.sql.functions import col, regexp_extract, current_timestamp

df_flat = df_raw.select(
    col("latitude"),
    col("longitude"),
    col("elevation"),
    col("timezone"),
    col("current.time").alias("weather_time_pkt"),
    col("current.temperature_2m").alias("temperature_c"),
    col("current.relative_humidity_2m").alias("humidity_pct"),
    col("current.apparent_temperature").alias("feels_like_c"),
    col("current.precipitation").alias("precipitation_mm"),
    col("current.weathercode").alias("weather_code"),
    col("current.pressure_msl").alias("pressure_hpa"),
    col("current.wind_speed_10m").alias("windspeed_kmh"),
    col("current.wind_direction_10m").alias("winddirection_deg"),
    col("current.cloud_cover").alias("cloud_cover_pct"),
    col("current.is_day").alias("is_day"),
    col("_metadata.file_path").alias("source_file"),
    regexp_extract(col("_metadata.file_path"), r'([A-Za-z]+)_\d{4}-\d{2}-\d{2}', 1).alias("city"),
    current_timestamp().alias("loading_time")
)

df_flat.show(truncate=False)

## Creating table weather_curated pointing to processed folder

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS internship_databricks_ws.default.weather_curated
USING DELTA
LOCATION '{processed_path}weather_curated/'
""")



## Appending new data to weather_curated

In [0]:
df_flat.write.format("delta").mode("append").option("overwriteSchema", "true").saveAsTable('internship_databricks_ws.default.weather_curated')
print("Appended new run to processed data")

## Dumps old .json files into Done folder

In [0]:

files_to_move = df_raw.select("_metadata.file_path").distinct().collect()

for row in files_to_move:
    source_path = row["file_path"]
    file_name = source_path.split("/")[-1]
    destination_path = f"{raw_path}Done/{file_name}"
    dbutils.fs.mv(source_path, destination_path)
    print(f"Moved: {file_name}")